In [3]:

import cv2
import numpy as np
import torch
import gradio as gr
from ultralytics import YOLO

# Load YOLOv5xu model with GPU acceleration if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model = YOLO("C:/Users/ASUS/Downloads/JUPYTER PROJECTS/RHPA PROJECT/yolov5xu.pt").to(device)

def process_video(input_video):
    """
    Process a video file, estimate the speed of moving animals, and generate an output video.
    """
    cap = cv2.VideoCapture(input_video)
    frame_width = int(cap.get(3))
    frame_height = int(cap.get(4))
    fps = int(cap.get(cv2.CAP_PROP_FPS))

    # Resize dimensions for faster processing (reduce resolution by 50%)
    frame_width_resized = frame_width // 2
    frame_height_resized = frame_height // 2
    scale_factor = 0.05  # Approximate meters per pixel (adjust based on real data)

    output_video = "processed_speed_video.mp4"
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video, fourcc, fps, (frame_width_resized, frame_height_resized))

    # Read first frame
    ret, prev_frame = cap.read()
    if not ret:
        cap.release()
        return "Error: Unable to read the video."

    prev_frame_resized = cv2.resize(prev_frame, (frame_width_resized, frame_height_resized))
    prev_gray = cv2.cvtColor(prev_frame_resized, cv2.COLOR_BGR2GRAY)

    frame_count = 0
    process_every_nth_frame = 3  # Run YOLO every 3rd frame for speed improvement

    while cap.isOpened():
        ret, next_frame = cap.read()
        if not ret:
            break

        next_frame_resized = cv2.resize(next_frame, (frame_width_resized, frame_height_resized))
        next_gray = cv2.cvtColor(next_frame_resized, cv2.COLOR_BGR2GRAY)

        # Compute Optical Flow (optimized parameters for speed)
        flow = cv2.calcOpticalFlowFarneback(prev_gray, next_gray, None, 0.5, 1, 3, 1, 5, 1.1, 0)
        movement = np.mean(np.abs(flow))

        # Convert to speed (meters per second)
        speed_mps = movement * fps * scale_factor
        speed_kph = speed_mps * 3.6  # Convert to km/h

        # Run YOLO detection only on every 3rd frame (speed optimization)
        if frame_count % process_every_nth_frame == 0:
            results = model(next_frame_resized)

            for result in results:
                boxes = result.boxes.xyxy.cpu().numpy()  # Get bounding boxes

                for box in boxes:
                    x_min, y_min, x_max, y_max = map(int, box)

                    # Draw bounding box
                    cv2.rectangle(next_frame_resized, (x_min, y_min), (x_max, y_max), (255, 0, 0), 2)

                    # Display speed on frame
                    label = f"Speed: {speed_kph:.2f} km/h"
                    cv2.putText(next_frame_resized, label, (x_min, y_min - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)

        # Write processed frame to output video
        out.write(next_frame_resized)

        # Update previous frame for next iteration
        prev_gray = next_gray
        frame_count += 1  # Increment frame counter

    cap.release()
    out.release()

    return output_video

# Create a Gradio interface
interface = gr.Interface(
    fn=process_video,
    inputs=gr.Video(label="Upload Video"),
    outputs=gr.Video(label="Processed Video"),
    title="Optimized Animal Speed Estimation",
    description="Upload a video to estimate the speed of moving animals using YOLOv5xu and Optical Flow."
)

# Launch the interface
interface.launch()



* Running on local URL:  http://127.0.0.1:7871

To create a public link, set `share=True` in `launch()`.



0: 416x640 (no detections), 553.3ms
Speed: 1.6ms preprocess, 553.3ms inference, 0.7ms postprocess per image at shape (1, 3, 416, 640)

0: 416x640 (no detections), 570.0ms
Speed: 1.4ms preprocess, 570.0ms inference, 0.5ms postprocess per image at shape (1, 3, 416, 640)

0: 416x640 (no detections), 544.0ms
Speed: 1.6ms preprocess, 544.0ms inference, 0.6ms postprocess per image at shape (1, 3, 416, 640)

0: 416x640 (no detections), 549.0ms
Speed: 1.4ms preprocess, 549.0ms inference, 0.5ms postprocess per image at shape (1, 3, 416, 640)

0: 416x640 (no detections), 560.7ms
Speed: 1.4ms preprocess, 560.7ms inference, 0.5ms postprocess per image at shape (1, 3, 416, 640)

0: 416x640 (no detections), 568.4ms
Speed: 1.2ms preprocess, 568.4ms inference, 0.5ms postprocess per image at shape (1, 3, 416, 640)

0: 416x640 (no detections), 577.3ms
Speed: 1.1ms preprocess, 577.3ms inference, 0.4ms postprocess per image at shape (1, 3, 416, 640)

0: 416x640 (no detections), 565.8ms
Speed: 1.3ms prepr